In [1]:
import pandas as pd
import numpy as np
import glob

In [2]:
#reading all CSV files at once to find total rows
files = glob.glob("/content/drive/MyDrive/cyclistic_case_study/working_data/*.csv")
dfs = [pd.read_csv(file) for file in files]

total_rows = 0
for i in range(0, len(dfs)):
  total_rows += len(dfs[i])

print(total_rows)

6351159


In [3]:
#check if columns name are consistant
reference_columns = list(dfs[0].columns)
inconsistant_columns = []
for i in range(0,len(dfs)):
  if list(dfs[i].columns) != reference_columns:
    inconsistant_columns.append(list(dfs[i].columns))

print(inconsistant_columns)



[]


In [4]:
# concating all 13 csv into 1 csv
combined_df = pd.concat(dfs, axis=0, ignore_index=False)
print(combined_df.info(show_counts=True))

<class 'pandas.core.frame.DataFrame'>
Index: 6351159 entries, 0 to 653703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   ride_id             6351159 non-null  object 
 1   rideable_type       6351159 non-null  object 
 2   started_at          6351159 non-null  object 
 3   ended_at            6351159 non-null  object 
 4   start_station_name  4998132 non-null  object 
 5   start_station_id    4998132 non-null  object 
 6   end_station_name    4929389 non-null  object 
 7   end_station_id      4929389 non-null  object 
 8   start_lat           6351159 non-null  float64
 9   start_lng           6351159 non-null  float64
 10  end_lat             6344712 non-null  float64
 11  end_lng             6344712 non-null  float64
 12  member_casual       6351159 non-null  object 
dtypes: float64(4), object(9)
memory usage: 678.4+ MB
None


In [5]:
#data cleaning

#droping the null values from end_lat and end_lng
combined_df.dropna(subset=["end_lat", "end_lng"], inplace=True)
print(combined_df.info(show_counts=True))



<class 'pandas.core.frame.DataFrame'>
Index: 6344712 entries, 0 to 653703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   ride_id             6344712 non-null  object 
 1   rideable_type       6344712 non-null  object 
 2   started_at          6344712 non-null  object 
 3   ended_at            6344712 non-null  object 
 4   start_station_name  4991685 non-null  object 
 5   start_station_id    4991685 non-null  object 
 6   end_station_name    4929389 non-null  object 
 7   end_station_id      4929389 non-null  object 
 8   start_lat           6344712 non-null  float64
 9   start_lng           6344712 non-null  float64
 10  end_lat             6344712 non-null  float64
 11  end_lng             6344712 non-null  float64
 12  member_casual       6344712 non-null  object 
dtypes: float64(4), object(9)
memory usage: 677.7+ MB
None


In [6]:
#data cleaning

#creating new column that tracks whether station name entry is complete or missing
combined_df["station_name_status"] = np.where((combined_df["start_station_name"].isna() | combined_df["end_station_name"].isna()), "Missing", "Complete")



In [16]:
#data cleaning

#making started_at and ended_at into appropriate format
combined_df["started_at"] = pd.to_datetime(combined_df["started_at"], format="mixed")
combined_df["ended_at"] = pd.to_datetime(combined_df["ended_at"], format="mixed")

#reset index
combined_df.reset_index(drop=True, inplace=True)

#calculating ride length in minutes
combined_df["ride_length"] = round((combined_df["ended_at"] - combined_df["started_at"]).dt.total_seconds()/60)

#Find a particular day of week, months and hours from that timestamp and putting days data in data_of_week column
combined_df["day_of_week"] = combined_df["started_at"].dt.day_name()
combined_df['months'] = combined_df["started_at"].dt.month_name()
combined_df["hours"] = combined_df["started_at"].dt.hour


combined_df[["started_at", "ended_at", "ride_length", "hours", "day_of_week", "months"]].head()





,started_at,ended_at,ride_length,hours,day_of_week,months
0,2025-05-11 17:22:39.471,2025-05-11 18:11:19.249,49.0,17,Sunday,May
1,2025-05-05 08:02:09.251,2025-05-05 08:12:07.549,10.0,8,Monday,May
2,2025-05-02 10:32:33.062,2025-05-02 10:39:07.262,7.0,10,Friday,May
3,2025-05-12 11:12:16.579,2025-05-12 11:17:25.126,5.0,11,Monday,May
4,2025-05-01 10:13:36.821,2025-05-01 10:17:40.548,4.0,10,Thursday,May


In [28]:
#data cleaning

#details about duplicate data
duplicate_rows = combined_df[combined_df.duplicated]
duplicate_rows.info(show_counts=True)

#dropping the duplicate data based on ride_id
combined_df.drop_duplicates(subset=["ride_id"], inplace=True)


<class 'pandas.core.frame.DataFrame'>
Index: 29 entries, 5755720 to 6307499
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ride_id              29 non-null     object        
 1   rideable_type        29 non-null     object        
 2   started_at           29 non-null     datetime64[ns]
 3   ended_at             29 non-null     datetime64[ns]
 4   start_station_name   23 non-null     object        
 5   start_station_id     23 non-null     object        
 6   end_station_name     21 non-null     object        
 7   end_station_id       21 non-null     object        
 8   start_lat            29 non-null     float64       
 9   start_lng            29 non-null     float64       
 10  end_lat              29 non-null     float64       
 11  end_lng              29 non-null     float64       
 12  member_casual        29 non-null     object        
 13  station_name_status  29 non-nul

In [29]:
#data cleaning

#removing the rides length with negative value, zero, less than 1 min or more than 48 hours
combined_df.loc[(combined_df["ride_length"] <1) | (combined_df["ride_length"]>2880), "ride_length"] = pd.NA
combined_df.dropna(subset=["ride_length"], inplace=True)

In [31]:
#display information about the cleaned data
print(combined_df.info(show_counts=True))

#export the cleaned data as CSV file
combined_df.to_csv("cyclistic_ride_13_months_cleaned_data.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 6219779 entries, 0 to 6344711
Data columns (total 18 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   ride_id              6219779 non-null  object        
 1   rideable_type        6219779 non-null  object        
 2   started_at           6219779 non-null  datetime64[ns]
 3   ended_at             6219779 non-null  datetime64[ns]
 4   start_station_name   4942568 non-null  object        
 5   start_station_id     4942568 non-null  object        
 6   end_station_name     4901582 non-null  object        
 7   end_station_id       4901582 non-null  object        
 8   start_lat            6219779 non-null  float64       
 9   start_lng            6219779 non-null  float64       
 10  end_lat              6219779 non-null  float64       
 11  end_lng              6219779 non-null  float64       
 12  member_casual        6219779 non-null  object        
 13  st